# FUNCTIONS

In [8]:
import rasterio
from rasterio import plot
import matplotlib.pyplot as plt
import numpy as np
import os, glob
from array import array
import csv
import pandas as pd
import struct
import re
from numpy import asarray
from numpy import savetxt
import matplotlib
import math
import time
from  datetime import datetime
import matplotlib.animation as animation
from matplotlib.animation import FuncAnimation
import matplotlib.patheffects as path_effects
import geopandas


## Read bands

In [19]:
def read_bands(path, j, f):
    
    picList = glob.glob(os.path.join(path,"*.tif"))
    
    if picList == []:
        picList = glob.glob(os.path.join(path,"*.tiff"))
    
    if picList == []:
        print("Error, or simply no image in this directory.")
    
    # REMOVE THE PICTURES THAT ARE NOT PURE BANDS DATA (THERMAL, TRUE COLOR, ETC.)

    newpicList = []
    
    for i in range(len(picList)):
        
        if picList[i].find("olor") == -1 and picList[i].find("Thermal") == -1:
            newpicList.append(picList[i])
    
    number_bands = len(newpicList)
    #print(newpicList)        

    #id = ['B01', 'B02', 'B03', 'B04', 'B05', 'B06','B07','B08','B8A','B09','B11','B12']
    
    rast_dc = {}
    rast_dc_corr = {} # The reflectances above 1 are set to one to avoid influencing the plot
    
    newpicList.sort()
    
    for i, filename in enumerate(newpicList):
        
     
        # Give a variable name
        
        # Sentinel-2_L2A
        start = 'Sentinel-2_L2A_' 
        end = '_(Raw).tif'
        actual_band = picList[i]
        
        #print("Actual band:", actual_band)
        
        if actual_band.find(start) == -1:
            start = "Sentinel_2_L2A_"
            
            if actual_band.find(start) == -1:
                start = "Landsat_8-9_L2_"
                
                if actual_band.find(start) == -1:
                    print("Further error with the initiator satellite name. Check it")
            
        if actual_band.rfind(end) == -1:
            end = "_Raw_.tif"
            
            if actual_band.rfind(end) == -1:
                end = "__Raw__"
                
                if actual_band.rfind(end) == -1:
                    end = "_(Raw)"
                    
                    if actual_band.rfind(end) == -1:    
                        print("Further error with the end format name. Check it")
            
            
        var_name = actual_band[actual_band.find(start) + len(start):actual_band.rfind(end)] 

        
        
        #print("actual_band.rfind(end):",actual_band.rfind(end))
        
        #print("PicList:", picList[i])
        #print("Variable name:",var_name)
        data = rasterio.open(picList[i])
        rast = data.read(1)
        #rast_dc[var_name] = data.read(1)
        #true_rast_size = np.count_nonzero(rast!=-1)
        #rast[rast == -1] = float('nan')
        rast_dc[var_name] = rast
        #print(rast)
        
        #no_zeros = rast[np.where(rast==0)]
        #print(f"number of zeros:{no_zeros.size}")
        #print(f"fraction of zeros:{no_zeros.size/true_rast_size}")
        
        # Maximums and minimums of the raster
        
        #print("Min", var_name,":",np.amin(rast[rast>-1]))
        #print("Max", var_name,":",np.amax(rast))
        
        # Print the values and plot rasters of the actual band
        # print(rast)
        # plt.imshow(rast, cmap='Greys')
        # plt.show()
        """
        index = dumb_but_effective(var_name)    
        mean_val[index,j] = np.nanmean(rast)
        std_val[index,j] = np.nanstd(rast)
        median_val[index,j] = np.nanmedian(rast)
        """
        print(var_name)
        
        # Display the pixels values in histograms
        
        """
        binwidth = 0.01
        central_wvl = [443, 490, 560, 665, 705, 740, 783, 842, 865, 940, 1610, 2190] #1375 for B10 does not appear
        xticks = ["B01 - 443", "B02 - 490", "B03 - 560","B04 - 665", "B05 - 705", "B06 - 740", "B07 - 783", "BO8 - 842", "B8A - 865", "B09 - 940", "B11 - 1610", "B12 - 2190"]
        plt.figure(data_hist)
        plt.subplot(f,number_bands + 1, dumb_but_effective(var_name) + 2 + j*(number_bands + 1))
        
        
        clean_rast = rast[rast > -1]
        bin_height, bin_boundary = np.histogram(clean_rast, bins = np.arange(0,1.5+binwidth,binwidth))
        bin_height = bin_height/float(max(bin_height))
        plt.bar(bin_boundary[:-1], bin_height, binwidth)
        plt.rcParams['axes.linewidth'] = 1

        """
        # Alternative to histograms (obsolete)
        
        """
        central_wvl = [443, 490, 560, 665, 705, 740, 783, 842, 865, 940, 1610, 2190] #1375 for B10 does not appear
        xticks = ["B01 - 443", "B02 - 490", "B03 - 560","B04 - 665", "B05 - 705", "B06 - 740", "B07 - 783", "BO8 - 842", "B8A - 865", "B09 - 940", "B11 - 1610", "B12 - 2190"]
        plt.figure(data_hist)
        plt.subplot(N_b,12, i + 1 + j*12)
        clean_rast = rast[rast > -1]
        plt.hist(clean_rast, bins = np.arange(0, 1.5 + binwidth, binwidth), normed = 1)
        #plt.bar(mean_val[index,j], 200000, color = 'red')
        #plt.bar()
        """
        #plt.xticks(fontsize = 1)
        #plt.yticks(fontsize = 1)
        #plt.show()
        #print("Index:", index)
        #print("Mean pixel value of band", var_name, ":",mean_val[index,j],"\n")
        
        rast2 = rast
        rast2[rast2 >1 ] = 1
        rast_dc_corr[var_name] = rast2



    return rast_dc, rast_dc_corr, number_bands

## Read and store bands

In [3]:
def read_and_store(path, j,f):

    # TAKE A
    picList = glob.glob(os.path.join(path,"*.tif"))
    
    if picList == []:
        picList = glob.glob(os.path.join(path,"*.tiff"))
    
    if picList == []:
        print("Error, or simply no image in this directory.")
    
    # REMOVE THE PICTURES THAT ARE NOT PURE BANDS DATA (THERMAL, TRUE COLOR, ETC.)

    newpicList = []
    
    for i in range(len(picList)):
        
        if picList[i].find("olor") == -1 and picList[i].find("Thermal") == -1 and picList[i].find("classification") == -1:
            newpicList.append(picList[i])
    
    number_bands = len(newpicList)
    #print(newpicList)        

    #id = ['B01', 'B02', 'B03', 'B04', 'B05', 'B06','B07','B08','B8A','B09','B11','B12']
    
    newpicList.sort()
    
    for i, filename in enumerate(newpicList):
        
     
        # Give a variable name
        
        # Sentinel-2_L2A
        start = 'Sentinel-2_L2A_' 
        end = '_(Raw).tif'
        actual_band = picList[i]
        
        #print("Actual band:", actual_band)
        
        if actual_band.find(start) == -1:
            start = "Sentinel_2_L2A_"
            
            if actual_band.find(start) == -1:
                start = "Landsat_8-9_L2_"
                
                if actual_band.find(start) == -1:
                    print("Further error with the initiator satellite name. Check it")
            
        if actual_band.rfind(end) == -1:
            end = "_Raw_.tif"
            
            if actual_band.rfind(end) == -1:
                end = "__Raw__"
                
                if actual_band.rfind(end) == -1:
                    end = "_(Raw)"
                    
                    if actual_band.rfind(end) == -1:    
                        print("Further error with the end format name. Check it")
            
            
        var_name_long = actual_band[actual_band.find(start) + len(start):actual_band.rfind(end)] +"_" +os.path.basename(path)
        var_name_short = os.path.basename(path)
        var_name = actual_band[actual_band.find(start) + len(start):actual_band.rfind(end)]
        
        # Open, read, remove the values out of the mask if needed, and store them in the dictionary
        data[var_name_long] = rasterio.open(picList[i])
        rast = data[var_name_long].read(1)
        rast[rast == -1] = float('nan')
        rast_dc[var_name_long] = rast


        # Get the transformations (useful for georeferenced plot)

        if i == 0: # Only for the first iteration, supposed equal for the bands of an image
            bounds_dc[var_name_short] = data[var_name_long].bounds

        
        
        # Maximums and minimums of the raster
        
        #print("Min", var_name,":",np.amin(rast[rast>-1]))
        #print("Max", var_name,":",np.amax(rast))
        
        index = dumb_but_effective(var_name)    
        mean_val[index,j] = np.nanmean(rast)
        std_val[index,j] = np.nanstd(rast)
        median_val[index,j] = np.nanmedian(rast)

        # SCALING OF THE REFLECTANCES BETWEEN 0 AND 1 --> warning to the temperature !
        """
        rast2 = rast
        rast2[rast2 >1 ] = 1
        rast2[rast2 <0] = 0
        rast_dc_corr[var_name] = rast2
        """
        if var_name == 'B10':

            if glob.glob(os.path.join(path,"*.shp")) == []:
                continue
                
            shapefile_path = glob.glob(os.path.join(path,"*.shp"))[0]
            shapefile = gpd.read_file(shapefile_path)

            geotiff_path = newpicList[i]
            print(geotiff_path)
            geotiff = rasterio.open(geotiff_path)

            bounds = bounds_dc[var_name_short]
            fig, ax = plt.subplots()
            show = rasterio.plot.show(geotiff, ax=ax, extent = (bounds.left, bounds.right, bounds.bottom, bounds.top), cmap ='Blues')
            # Add colorbar

            cax = fig.add_axes([0.85,0.1,0.03,0.8])
            colorbar = Colorbar(ax=cax, mappable = ax.images[0], orientation = 'vertical')
            #colorbar.set_label("Colorbar Label")

            shapefile.plot(ax=ax,facecolor='none',edgecolor='red')
            plt.show()
        

    return rast_dc, rast_dc_corr, number_bands

In [18]:
def dumb_but_effective(var_name):
    
    global index
    
    if var_name == 'B01':
        index = 0
    if var_name == 'B02':
        index = 1
    if var_name == 'B03':
        index = 2
    if var_name == 'B04':
        index = 3
    if var_name == 'B05':
        index = 4
    if var_name == 'B06':
        index = 5
    if var_name == 'B07':
        index = 6
        
    # For Sentinel
    if var_name == 'B08':
        index = 7
    if var_name == 'B8A':
        index = 8
    if var_name == 'B09':
        index = 9
    if var_name == 'B11':
        index = 10
    if var_name == 'B12':
        index = 11
        
    # For Landsat
    if var_name == 'B10':
        index = 7
        
    return index

## Plot normalized SWIR

## Plot SWIR (bands 12, 8A, 4)

In [11]:
def plot_SWIR(fold, path, data, bounds_dc):
    
    B04_var = 'B04_' + fold
    B8A_var = 'B8A_' + fold
    B12_var = 'B12_' + fold

    B04_path = data[B04_var].name
    B8A_path = data[B8A_var].name
    B12_path = data[B12_var].name
    
    B04_dataset = rasterio.open(B04_path)
    B8A_dataset = rasterio.open(B8A_path)
    B12_dataset = rasterio.open(B12_path)

    B04_band = B04_dataset.read(1)
    B8A_band = B8A_dataset.read(1)
    B12_band = B12_dataset.read(1)

    SWIR_image = np.dstack((B12_band, B8A_band, B04_band))

    if glob.glob(os.path.join(path,"*full_thickness.shp")) != []:

        shapefile_path = glob.glob(os.path.join(path,"*full_thickness.shp"))[0]
        shapefile = gpd.read_file(shapefile_path)
        
        bounds = bounds_dc[fold]
        extent = (bounds.left, bounds.right, bounds.bottom, bounds.top)
        
        fig,ax = plt.subplots()
        plt.imshow(SWIR_image, extent=extent)
        shapefile.plot(ax=ax,facecolor='none',edgecolor='red')
        plt.savefig(save_path + 'normal_SWIR_' + fold + '.tiff')
        plt.show()


In [ ]:
def days(path,date):
    
    s1 = os.path.basename(path)
    s2 = "20" + s1[5:]
    date = date.append(datetime.strptime(s2,'%Y_%m_%d'))
    
    return datetime